# Module 07 — Notebook 2: Package Management

## Learning Objectives

By the end of this notebook, you will be able to:

- Install and uninstall packages with `uv pip` and `pip`
- Understand version specifiers: `==`, `>=`, `~=`
- Generate a `requirements.txt` with `uv pip freeze`
- Install from a `requirements.txt`
- Parse and validate a `requirements.txt` file using Python
- Understand when to use `pyproject.toml` instead

**Estimated time:** ~20 minutes

## Why This Matters for AI Research Engineering

Reproducibility in ML papers requires exact package versions. NumPy's random number generation, scikit-learn's default hyperparameters, and even floating-point results can differ across package versions. A `requirements.txt` is a contract: *"these numbers came from this exact environment."*

When someone says "I can't reproduce your results," the first question is usually "Did you use the same package versions?"

In [ ]:
import sys
sys.path.insert(0, "../../")
from src.checks import check_equal, check_type, check_contains, check_length, check_keys
import importlib.metadata
from pathlib import Path
print("Setup complete.")

## 1. Installing Packages

The JS equivalent of `npm install` is `uv pip install` (or `pip install`). Run these in your terminal after activating your venv.

```bash
# Modern (uv — much faster than pip):
uv pip install requests
uv pip install "numpy>=1.24,<2.0"

# Classic (built-in, always available):
pip install requests
pip install "numpy>=1.24,<2.0"

# Multiple packages at once:
uv pip install numpy pandas matplotlib

# Install your own local project in editable mode:
uv pip install -e .

# Uninstall:
uv pip uninstall requests
```

All of these only affect the active virtual environment — not your system Python.

## 2. Version Specifiers

Version specifiers tell the installer which versions are acceptable. For **reproducibility**, always use `==` in requirements files.

| Specifier | Meaning | Example |
|---|---|---|
| `==1.26.0` | Exactly this version | Always installs 1.26.0 |
| `>=1.24` | This version or newer | 1.24, 1.25, 1.26 ... |
| `<=2.0` | This version or older | Anything up to 2.0 |
| `~=1.26.0` | Compatible release | `>=1.26.0, <1.27` |
| `>=1.24,<2.0` | Range | Any 1.x at or after 1.24 |

JS comparison:
- `^1.2.3` in npm ≈ `~=1.2.3` in Python (allows patch/minor bumps)
- `1.2.3` (exact) in npm ≈ `==1.2.3` in Python

For AI research: **use `==` in any requirements file that others need to reproduce your work.**

## 3. requirements.txt — Pinning Your Environment

A `requirements.txt` lists exact package versions. You generate it with `freeze`, which outputs the current environment's package list:

```bash
# Generate requirements.txt from your current environment:
uv pip freeze > requirements.txt

# Or with pip:
pip freeze > requirements.txt

# Install from requirements.txt (in a fresh venv):
uv pip install -r requirements.txt
pip install -r requirements.txt
```

The file looks like this:

In [ ]:
%%writefile example_requirements.txt
# Core numeric packages
numpy==1.26.4
pandas==2.2.0

# Visualization
matplotlib==3.8.3

# Machine learning
scikit-learn==1.4.0

In [ ]:
# Read and parse it in Python
req_text = Path("example_requirements.txt").read_text()

pinned = []
for line in req_text.strip().splitlines():
    line = line.strip()
    if line and not line.startswith("#"):
        pinned.append(line)

print(f"Pinned packages ({len(pinned)}):")
for p in pinned:
    print(f"  {p}")

## 4. pyproject.toml — The Modern Project File

`pyproject.toml` is the modern Python standard for project configuration (like `package.json`). It serves a different purpose than `requirements.txt`:

| | `requirements.txt` | `pyproject.toml` |
|---|---|---|
| **Use case** | Pin an analysis environment snapshot | Define a distributable Python package |
| **Version style** | Always `==` (exact) | Often `>=` (floor) |
| **Who uses it** | Anyone running your analysis | Users installing your library |

This repo uses `pyproject.toml` because it's a proper Python package. For a one-off analysis, `requirements.txt` is usually the right choice.

```toml
[project]
name = "my-analysis"
version = "0.1.0"
requires-python = ">=3.11"
dependencies = [
    "numpy>=1.24",
    "pandas>=2.0",
]
```

## Exercise 1 — Parse a requirements.txt

Parse `sample_reqs_text` into a list of package strings. Skip blank lines and comment lines (lines starting with `#`).

In [ ]:
sample_reqs_text = """
# Core numeric packages
numpy==1.26.4
pandas==2.2.0

# Visualization
matplotlib==3.8.3
"""

# YOUR CODE HERE
# Build a list of package strings by iterating over lines,
# stripping whitespace, and skipping blank lines and comments.
parsed_packages = None  # list of 3 strings like "numpy==1.26.4"

In [ ]:
check_type(parsed_packages, list, "parsed_packages is a list")
check_length(parsed_packages, 3, "3 packages parsed (skipping blanks and comments)")
check_contains(parsed_packages, "numpy==1.26.4", "numpy entry is present")
check_contains(parsed_packages, "matplotlib==3.8.3", "matplotlib entry is present")

## Exercise 2 — Extract Package Names and Versions

From `parsed_packages` (Exercise 1), build a dict mapping `package_name → version_string`.

Expected result: `{"numpy": "1.26.4", "pandas": "2.2.0", "matplotlib": "3.8.3"}`

In [ ]:
# YOUR CODE HERE
# Hint: each entry in parsed_packages looks like "name==version"
# Split on "==" to get [name, version]
pkg_versions = None  # dict: {str: str}

In [ ]:
check_type(pkg_versions, dict, "pkg_versions is a dict")
check_keys(pkg_versions, ["numpy", "pandas", "matplotlib"], "correct package names as keys")
check_equal(pkg_versions["numpy"], "1.26.4", "numpy version is 1.26.4")
check_equal(pkg_versions["matplotlib"], "3.8.3", "matplotlib version is 3.8.3")

## Exercise 3 — Write Your Own requirements.txt

Create `my_requirements.txt` with at least 3 packages pinned using `==`. Include numpy, pandas, and at least one more.

You can find the exact installed versions with:
```python
import importlib.metadata
importlib.metadata.version("numpy")  # e.g. "1.26.4"
```

In [ ]:
# Run this to find exact versions for your requirements.txt
import importlib.metadata
for pkg in ["numpy", "pandas", "matplotlib", "scikit-learn"]:
    try:
        print(f"{pkg}=={importlib.metadata.version(pkg)}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{pkg} — not installed")

In [ ]:
%%writefile my_requirements.txt
# YOUR REQUIREMENTS HERE
# List at least 3 packages with exact == versions
# Include numpy and pandas


In [ ]:
req_path = Path("my_requirements.txt")
check_equal(req_path.exists(), True, "my_requirements.txt was created")

content = req_path.read_text()
check_contains(content, "numpy==", "numpy is pinned with ==")
check_contains(content, "pandas==", "pandas is pinned with ==")

pinned_lines = [l for l in content.splitlines() if "==" in l]
check_equal(len(pinned_lines) >= 3, True, "at least 3 packages are pinned with ==")

## Wrap-Up

| Command / Code | What it does |
|---|---|
| `uv pip install pkg==1.2.3` | Install an exact version |
| `uv pip list` | List installed packages |
| `uv pip freeze > requirements.txt` | Pin current environment |
| `uv pip install -r requirements.txt` | Install from a pin file |
| `importlib.metadata.version("pkg")` | Check installed version in Python |
| `==` specifier | Pin to an exact version (use this in requirements files) |
| `>=` specifier | Accept this version or newer (use in pyproject.toml dependencies) |

**Next:** Notebook 3 — Seeds and Reproducibility (making stochastic code deterministic)